# Assignment 4: LM Alignment

- Before running the jupyter notebook, don't forget to copy it into your drive **(`File` => `Save a copy in Drive`)**. *Failing to do this step may result in losing the progress of your code.*
- **Don't forget to choose "Runtime Type" = GPU in Colab for running this notebook (Runtime > Change Runtime Type > T4 GPU).**

- **You will do the following task as detailed under each section, and we will grade:**
  - **Coding Exercises:** You will complete the the code blocks denoted by **`TODO:`**. We will grade your code in this notebook through the autograder.

- We make the write-up into reflective questions for you to explore your implementation and understand the topics better. You **are not required** to submit your answers to these questions.

*Adapted from course material of Winter 2024 CSE 447/517 instructed by Yejin Choi at University of Washington, with feedback from Yichi Yang, Xinyu ``Frederick'' Pi, and Lianhui Qin. It was originally designed by Taylor Sorensen, Skyler Hallinan, Melanie Sclar, Alisa Liu, and Liwei Jiang with invaluable feedback from Yejin Choi.*

# Section 1: Setup and Baseline Evaluation


### 1.0 Preparation

In [ ]:
! pip install transformers==4.38
! pip install datasets

In [ ]:
import torch
import random
from tqdm import tqdm
import torch.nn.functional as F
from datasets import load_dataset
from matplotlib import pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from transformers import AdamW, set_seed
from typing import Dict, Union, List, Tuple

## **Coding Exercises** for Section 1:
**You will complete the following code blocks denoted by `TODO:`.**

### 1.1 Loading the base model, the reward model, and data

In [ ]:
"""
Load initial model and tokenizer and send it to GPU if available.

Since you will be batching and GPT2 is a decoder-only architecture,
load your tokenizer with padding_side='left' to ensure
that all the <pad> tokens appear before the tokenized text.

To avoid warnings, consider setting pad_token to its eos_token.
"""

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('Device:', device)
model_name = 'distilgpt2'

# TODO: Load the tokenizer.
# Hint: remember to add paddings on the left for the decoder-only model.
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side = 'left')

# TODO: set the pad_token to the eos_token.

# TODO: Load the model and sent it to the device.
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

model.config.pad_token_id = tokenizer.eos_token_id

In [ ]:
"""
Load the reward model.
"""

# TODO: Load reward model and tokenizer from huggingface (use "omidroshani/imdb-sentiment-analysis").
# Hint: Remember to send the model to the device.
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased")

In [1]:
"""
Load data.
"""

# TODO: Load IMDB dataset (https://huggingface.co/datasets/imdb) train & test splits.
dataset = load_dataset('https://huggingface.co/datasets/imdb')
train_texts = dataset['train']
test_texts = dataset['test']

random.seed(42)

# TODO: shuffle the samples within every data split.
train_texts = train_texts.shuffle()
test_texts = test_texts.shuffle()

SyntaxError: invalid syntax (<ipython-input-1-a77942d13558>, line 6)

### 1.2 Evaluation.

In [ ]:
def compute_reward_for_mini_batch(batch,
                                  tokenizer, model,
                                  reward_tokenizer, reward_model,
                                  seed_token_length=10, max_new_tokens=20):
  """
  Write an evaluation function that for each review in the test set it
    1. tokenizes the review.
    2. truncates the review to 10 tokens (seed_token_length).
    3. Given the truncated seed review, randomly generate 20 more tokens with
       the LM so that the final review has a length of 30 tokens.
    4. Get the probability of the generated review being positive.

  The evaluation function should return the final average rewards and the text generations.
  """
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  # TODO: tokenize the review, truncating it to 10 tokens (i.e., seed_token_length).
  # Hint: remember to send the encoded text to the device.
  input_tokens = tokenizer(batch, return_tensors="pt")[:seed_token_length]

  # TODO: randomly generate 20 more tokens (i.e., max_new_tokens) with the LM so
  # that the final review has a length of 30 tokens.
  output_tokens = 

  # TODO: decode the generated texts and store them in a list.
  generated =

  # TODO: get the probability of the generated review being positive.
  # Hint: you will need to encode the generated text, feed it through the reward model,
  # and get the probablity of the output scores.
  reward =

  return reward, output_tokens, generated


def evaluate(tokenizer, model, reward_tokenizer, reward_model, batch_size=32, verbose=False):
  """
  Code an evaluation loop that simultaneously evaluates a mini-batch
  of size batch_size. Use the helper function compute_reward_for_mini_batch().
  """
  set_seed(42)
  with torch.no_grad():
    test_batches = len(test_texts) // batch_size

    # Evaluate the whole dataset by looping through
    # mini-batch reward computations and accumulate total_reward.
    total_reward = 0
    generations = []
    loop = tqdm(total=test_batches, position=0, leave=False)
    for i in range(0, len(test_texts), batch_size):
        # TODO: get the current batch of data.
        batch =

        # TODO: use compute_reward_for_mini_batch() for evaluating each mini batch.
        reward, _, generated =

        # TODO: add generated to generations.

        # TODO: add average batch reward to total_reward.

        # TODO: compute the average_reward so far for the display of the progress bar.
        average_reward =

        loop.set_description(f"Average Reward: {average_reward:.4f}")
        loop.update(1)

    # TODO: compute the final_average_reward.
    final_average_reward =
    if verbose:
      print(f"Final Average Reward: {final_average_reward:.4f}")
      print(f"Example texts: {generations[:5]}")

    return {
        'reward': final_average_reward,
        'generations': generations,
    }


In [ ]:
results = evaluate(tokenizer, model, reward_tokenizer, reward_model, verbose=True)
print("Reward:", results["reward"])

## **Reflective Questions** for Section 1:

---

**Q1.1:** Run the pretrained model on the evaluation function over the entire test set. What is the average reward?

**Hint:** On a free [Google Colab](https://colab.google/) T4 instance, with a generation batch size of 32, this should take about 5 minutes to run.



**Q1.2:** Check out 5 example generations from the model. How do you feel about the quality of the generations? What sentiment do they express?


# Section 2: Implementing REINFORCE

## **Coding Exercises** for Section 2:
**You will complete the following code blocks denoted by `TODO:`.**



### 2.1 Implement the REINFORCE loss

In [ ]:
def compute_reinforce_loss(reward, output_tokens, model):
  """
  Compute REINFORCE loss given the generations' rewards (reward) along with the
  models' generations (output_tokens) that we need to compute probabilities' over.

  Return the log probabilities of the output tokens, and the REINFORCE loss.
  """
  # TODO: get batch_size.
  batch_size =

  # TODO: feed output_tokens into the model, and get log_probs with output logits
  # using log_softmax.
  output = model(input_ids=output_tokens)
  log_probs = F.log_softmax(output.logits, dim=-1)

  # TODO: choose logprobs of actual generated tokens.
  # Hint: note that output_tokens contains the 10 seed tokens (token 1 ... 10)
  # and the 20 continuation tokens (token 11 ... 30) generated by the model.
  # To get log_probs, putting output_tokens through the model will condition on
  # token 1 ... 30 to predict the next token (at position 31) in the sequence.
  # Therefore, log_probs contains the log probs of tokens at position 2 ... 31.
  # We want to choose the log_probs of tokens at position 2 ... 30 for
  # computing the REINFORCE loss.
  # Hint: chosen_log_probs should have size: torch.Size([batch_size, 29]).
  # Hint: you may find this example code helpful, https://github.com/huggingface/transformers/blob/d90acc16437e8c9e45e068fa1cc1a263b9a7208f/src/transformers/models/gpt2/modeling_gpt2.py#L1103C13-L1103C63
  chosen_log_probs =

  # TODO: compute batched REINFORCE loss with chosen_log_probs and reward.
  # Hint: it can be helpful to reshape reward for batched computation.
  batch_loss =

  # TODO: average the batch_loss to get the final REINFORCE loss.
  loss =

  return log_probs, loss

### 2.2 REINFORCE training

In [ ]:
######################################################
#  The following helper function is given to you.
######################################################
def reset_model_optimizer(model_name, tokenizer, lr=1e-4, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    model.config.pad_token_id = tokenizer.eos_token_id
    optimizer = AdamW(model.parameters(), lr=lr)
    return model, optimizer

In [ ]:
def train(model_name, tokenizer, reward_tokenizer, reward_model, train_texts,
          alpha=.001, lr=1e-4, batch_size=32, use_kl_divergence=False, debug=False, device=None):
  """
    Train model by computing loss for each mini-batch, and update weights as
    needed using the optimizer.

    For computing the mini-batch loss,
    1. Compute rewards,
    2. Compute REINFORCE loss,
    3. Include KL divergence loss only if use_kl_divergence=True.
  """
  set_seed(42)

  if device is None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  model, optimizer = reset_model_optimizer(model_name, tokenizer, lr=lr, device=device)

  train_samples = train_texts[:50] if debug else train_texts

  if use_kl_divergence:
    kl_divergences = []
    orig_model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

  losses = []
  rewards = []
  batches = len(train_samples) // batch_size
  loop = tqdm(total=batches, position=0, leave=False)
  for i in range(0, len(train_samples), batch_size):
    # TODO: get the current batch of data.
    batch =

    # TODO: compute reward of the batch using compute_reward_for_mini_batch().
    # Hint: you need to detach the reward tensor to break gradient updates.
    reward, output_tokens, _ =
    reward =

    # TODO: compute REINFORCE loss.
    log_probs, loss =

    # TODO: Compute kl divergence, and modify loss accordingly with alpha.
    # You will complete this for section 3.
    if use_kl_divergence:
      pass

    # TODO: compute gradients and update parameters using optimizer.

    # TODO: add mean reward, loss, and kl_divergence to their corresponding lists,
    # when appropriate.

    if use_kl_divergence:
      loop.set_description(f"Loss: {loss.item():.4f}, KL: {kl_divergence.item():.4f}, Reward: {reward.mean().item():.4f}")
    else:
      loop.set_description(f"Loss: {loss.item():.4f}, Reward: {reward.mean().item():.4f}")
    loop.update(1)

  if debug:
    ### skip evaluate if in debugging mode
    eval_results = []
  else:
    eval_results = evaluate(tokenizer, model, reward_tokenizer, reward_model)

  return_dict = {
      'losses': losses,
      'rewards': rewards,
      'eval_results': eval_results
  }
  if use_kl_divergence:
    return_dict['kl_divergences'] = kl_divergences

  return return_dict



In [ ]:
### Train with REINFORCE
### Set "debug" to False to run the full training loop
results_reinforce = train(model_name, tokenizer, reward_tokenizer, reward_model, train_texts, debug=True, use_kl_divergence=False)
print("Losses:", results_reinforce['losses'])

### 2.3 Plot

In [ ]:
plt.plot(results_reinforce['rewards'])
plt.xlabel('# batch')
plt.ylabel('reward')

## **Reflective Questions** for Section 2:
Consider the following questions to help you debug and deepen your understanding of your REINFORCE implementation.

---

**Q2.1:** After running your training function for one epoch, what patterns do you observe in the reward over time?



**Q2.2:** Examine a few sample generations from your model. How do these compare to the original reviews in terms of positivity?



**Q2.3:** Based on your observations of the generated reviews, what limitations of the REINFORCE algorithm become apparent?


# Section 3: Regularization

## **Coding Exercises** for Section 3:
**You will complete the following code blocks denoted by `TODO:`.**



### 3.1 Compute KL divergence

In [ ]:
def compute_kl_divergence(log_probs_current, output_tokens, orig_model):
  """
  Compute the KL divergence between the original model' log probabilities for generating
  output_tokens, and the current model's log probabilities (log_probs_current)
  """
  # TODO:

  return kl_divergence

### 3.2 Experiment with different alpha for KL regularization

In [ ]:
results_reinforce_kl_div_alpha_zero = train(model_name, tokenizer, reward_tokenizer, reward_model, train_texts, debug=True, use_kl_divergence=True, alpha=0.000)

In [ ]:
plt.figure()
plt.plot(results_reinforce_kl_div_alpha_zero['rewards'])
plt.xlabel('# batch')
plt.ylabel('reward')

plt.figure()
plt.plot(results_reinforce_kl_div_alpha_zero['kl_divergences'])
plt.xlabel('# batch')
plt.ylabel('KL divergence')

## **Reflective Questions** for Section 3:

---

**Q3.1:**

You can begin by setting α=0 (effectively removing the KL penalty) for one epoch. Observe and plot the KL-divergence between the original model and your new model over time.

- What trends do you see in the KL-divergence during training?
- Why might such a trend be undesirable in practice?


**Q3.2:**

Experiment with different values of α to explore the impact of KL-regularization. For instance:

- You can try a *high* α that prevents the model from achieving a reward of, say, 0.65 or more.
- You can try a *low* α that leads to notably poor output quality.
- Then, find an α that yields a reasonably high reward (e.g. ≥ 0.8) while maintaining mostly natural text.

For each scenario, you may check:

- How does the KL-divergence evolve over time?
- How does the reward change relative to your original model?
- What do the sample generations look like, and how does their quality compare across different α values?
